## ML methods to detect mobility pattern

In [ ]:
from glob import glob
import pandas as pd
import geopandas as gpd
import os
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

In [ ]:
aaa = pd.read_csv("/nas/houce/UK_mobility/data_20251226_new/ML/England_ML.csv")
BASE_TIME_COLS = ['workplace_weighted_hours_today', 't1_weighted_hours_today', 't3_weighted_hours_today', 't5_weighted_hours_today', 't6_weighted_hours_today', 't8_weighted_hours_today','t9_weighted_hours_today']
BASE_FREQ_COLS = ['workplace_frequency','t1_frequency','t3_frequency','t5_frequency','t6_frequency','t8_frequency','t9_frequency']


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

df_2020 = aaa[(aaa['month'] == 202002) | (aaa['month'] == 202003)]
df_2021 = aaa[aaa['month'] == 202103]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================================
# 2. 数据预处理：堆叠 (Stacking)
# ==========================================
# 核心思想：把 2021 的数据放到 2020 下面，形成一个 2N 行的长表
# 这样训练出来的聚类中心对两年都是通用的

# 提取数据
X_2020 = df_2020[BASE_TIME_COLS]
X_2021 = df_2021[BASE_TIME_COLS]

# 重命名列名，使其一致（去除年份后缀），以便堆叠
generic_cols = ['workplace', 't1', 't3', 't5', 't6', 't8', 't9']
X_2020.columns = generic_cols
X_2021.columns = generic_cols

# 拼接数据 (注意保留原始索引以便后续还原)
X_combined = pd.concat([X_2020, X_2021], axis=0)

# 标准化 (Clustering对尺度敏感，必须标准化)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

In [ ]:
from tqdm import tqdm
sse = []
K_range = range(3, 20)
for k in tqdm(K_range):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)

In [ ]:
x = np.array(list(K_range))
y = np.array(sse)

# 两端点
point1 = np.array([x[0], y[0]])
point2 = np.array([x[-1], y[-1]])

# 计算每个点到直线的距离
def distance(point, line_start, line_end):
    return np.abs(np.cross(line_end-line_start, line_start-point)) / np.linalg.norm(line_end-line_start)

distances = np.array([distance(np.array([xi, yi]), point1, point2) for xi, yi in zip(x, y)])
elbow_idx = np.argmax(distances)
elbow_k = x[elbow_idx]
elbow_sse = y[elbow_idx]

plt.figure(figsize=(6,4))
plt.plot(K_range, sse, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('SSE (Inertia)')
plt.title('Elbow Method For Optimal k')
plt.scatter(elbow_k, elbow_sse, color='red', s=100, label=f'Elbow at k={elbow_k}')
plt.legend()
plt.show()

In [ ]:
# ==========================================
# 3. 聚类算法 (以 K-Means 为例)
# ==========================================
# 假设聚成 9 类 (你可以通过肘部法则确定 K 值)
k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

# 在合并的大数据集上训练
kmeans.fit(X_scaled)

# 获取所有标签
all_labels = kmeans.labels_

# ==========================================
# 4. 标签还原与对齐
# ==========================================
# 将长长的标签数组切开，前一半是2020，后一半是2021
n = len(df_select)
labels_2020 = all_labels[:n]
labels_2021 = all_labels[n:]

# 将结果写回原始 DataFrame
df_select['cluster_2020'] = labels_2020
df_select['cluster_2021'] = labels_2021

# ==========================================
# 5. 结果分析：状态转移矩阵 (Transition Matrix)
# ==========================================
# 查看同一个 ID 从 2020 年的某一类 变成了 2021 年的哪一类
transition_matrix = pd.crosstab(df_select['cluster_2020'], df_select['cluster_2021'], 
                                rownames=['2020 Cluster'], colnames=['2021 Cluster'])

print("--- 状态转移矩阵 (行是2020类别，列是2021类别) ---")
print(transition_matrix)

# 计算具体的转移情况 (例如：保持不变的比例)
df_select['is_changed'] = df_select['cluster_2020'] != df_select['cluster_2021']
change_rate = df_select['is_changed'].mean()
print(f"\n发生类别变更的样本比例: {change_rate:.2%}")

In [ ]:
df_select

In [ ]:
from sklearn.decomposition import PCA
import umap

# 取2020和2021的特征
X_2020 = df_select[[i+'_ratio_2020' for i in cols_2020[3:]]].values
X_2021 = df_select[[i+'_ratio_2021' for i in cols_2021[3:]]].values

# PCA降维到2维
pca = PCA(n_components=2)
X_2020_pca = pca.fit_transform(X_2020)
X_2021_pca = pca.fit_transform(X_2021)


# 可视化
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].scatter(X_2020_pca[:, 0], X_2020_pca[:, 1], c=df_select['cluster_2020'], cmap='tab10', s=2, alpha=0.5)
axs[0].set_title('2020 PCA')
for cluster_id in np.unique(df_select['cluster_2020']):
    mask = df_select['cluster_2020'] == cluster_id
    axs[0].scatter(X_2020_pca[mask, 0], X_2020_pca[mask, 1], 
                   label=f'Cluster {cluster_id}', s=2, alpha=0.3, cmap='tab10')
axs[0].legend(title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')

axs[1].scatter(X_2021_pca[:, 0], X_2021_pca[:, 1], c=df_select['cluster_2021'], cmap='tab10', s=2, alpha=0.5)
axs[1].set_title('2021 PCA')
axs[1].legend(*axs[1].get_legend_handles_labels(), title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
for cluster_id in np.unique(df_select['cluster_2020']):
    mask = df_select['cluster_2020'] == cluster_id
    axs[1].scatter(X_2020_pca[mask, 0], X_2020_pca[mask, 1], 
                   label=f'Cluster {cluster_id}', s=2, alpha=0.3 , cmap='tab10')
axs[1].legend(title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.heatmap(transition_matrix, annot=True, fmt='d', cmap='Blues')

In [ ]:
transition_percent = transition_matrix.div(transition_matrix.sum(axis=1), axis=0) * 100
transition_percent = transition_percent.round(2)
sns.heatmap(transition_percent, annot=True, fmt='.2f', cmap='Blues')

In [ ]:
# 计算类别发生转变的总百分比（非对角线元素之和）
changed_percent = 100 - transition_percent.values.diagonal().sum() / transition_percent.values.sum() * 100
print(f"类别发生转变的用户百分比约为: {changed_percent:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def analyze_specific_cluster(df, target_cluster=1, year='2021'):
    """
    分析特定 Cluster 的成分，并与全局平均值对比
    """
    # 1. 准备特征列名
    feature_cols = [
                    f'days_t1_{year}', f'days_t3_{year}', f'days_t5_{year}', 
                    f'days_t6_{year}', f'days_t8_{year}', f'days_t9_{year}']
    
    # 简化列名以便绘图 (去掉年份后缀)
    readable_cols = ['t1 (Eating)', 't3 (Attraction)', 't5 (Health)', 
                     't6 (Infra)', 't8 (Retail)', 't9 (Transport)']
    
    # 2. 计算分组统计量 (均值)
    cluster_means = df.groupby(f'cluster_{year}')[feature_cols].mean()
    cluster_means.columns = readable_cols
    
    # 获取全局平均值
    global_mean = df[feature_cols].mean()
    global_mean.index = readable_cols
    
    # 获取目标 Cluster (Cluster 1) 的均值
    target_mean = cluster_means.loc[target_cluster]
    
    # ==========================================
    # 可视化 1: 热力图 (直观展示 Cluster 1 在所有类中的定位)
    # ==========================================
    plt.figure(figsize=(10, 6))
    # 标准化热力图 (Z-Score)，让高低差异更明显
    sns.heatmap(cluster_means, annot=True, cmap='RdBu_r', center=0, fmt=".2f")
    plt.title(f'{year} Cluster Means Heatmap (Raw Values)', fontsize=14)
    plt.ylabel('Cluster ID')
    plt.show()

    # ==========================================
    # 可视化 2: 目标 Cluster vs 全局平均 (雷达图或柱状图)
    # ==========================================
    # 这里用双柱状图对比，最清晰
    comparison_df = pd.DataFrame({
        f'Cluster {target_cluster}': target_mean,
        'Global Average': global_mean
    })
    
    comparison_df.plot(kind='bar', figsize=(10, 5), color=['#1f77b4', '#d62728'])
    plt.title(f'Cluster {target_cluster} vs. Global Average ({year})', fontsize=14)
    plt.ylabel('Average Days')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

    # ==========================================
    # 3. 自动解读报告
    # ==========================================
    print(f"--- Cluster {target_cluster} 成分解剖报告 ---")
    
    # 判断活跃度
    total_activity_cluster = target_mean.sum()
    total_activity_global = global_mean.sum()
    
    print(f"1. 总体活跃度: Cluster {target_cluster} = {total_activity_cluster:.2f} vs 全局 = {total_activity_global:.2f}")
    
    if total_activity_cluster < total_activity_global * 0.6:
        print("   >> 结论: 这是一个【低活跃/沉睡】群体 (Low Activity Group)。")
    elif total_activity_cluster > total_activity_global * 1.4:
        print("   >> 结论: 这是一个【高活跃/超级】群体 (High Activity Group)。")
    else:
        print("   >> 结论: 这是一个【普通/平均】群体 (Average/Mainstream Group)。")
        
    print("\n2. 特征偏好:")
    # 找出该 Cluster 明显高于平均值的特征
    diff = (target_mean - global_mean) / global_mean
    distinct_features = diff[diff > 0.2].index.tolist()
    
    if distinct_features:
        print(f"   >> 相比大众，该群体显著偏好: {', '.join(distinct_features)}")
    else:
        print("   >> 该群体没有显著的特定偏好，各项指标均低于或接近平均值。")
    return cluster_means
# --- 运行分析 ---
# 假设你的数据叫 df
# 只要运行这行代码即可
cluster_means_2020 = analyze_specific_cluster(df_select, target_cluster=1, year='2020')
cluster_means_2021 = analyze_specific_cluster(df_select, target_cluster=1, year='2021')

In [ ]:
cluster_means_delta = cluster_means_2021 - cluster_means_2020
cluster_means_delta.plot(kind='bar', figsize=(10, 5))
plt.title(f'Cluster 1 Feature Changes from 2020 to 2021', fontsize=14)
plt.ylabel('Change in Average Days')
plt.legend(title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()